# Training prep 

In [ ]:
# imports for all modules under src/preprocessing/*.py
from importlib import import_module
from pathlib import Path
import sys

# Make project src importable from this notebook location
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Auto-import every preprocessing module and expose as globals
preprocessing_modules = {}
preprocessing_dir = src_path / "preprocessing"
for module_file in sorted(preprocessing_dir.glob("*.py")):
    if module_file.stem.startswith("_"):
        continue
    module = import_module(f"preprocessing.{module_file.stem}")
    preprocessing_modules[module_file.stem] = module
    globals()[module_file.stem] = module

print("Loaded preprocessing modules:", ", ".join(preprocessing_modules.keys()))

Loaded preprocessing modules: Accept, approvalDate, approvalFY, BalanceGross, base_cleaning, city_bank, createJob, DisbursementDate, DisbursementGross, example, franchise_code, LowDoc, newExists, noemp, retainedJob, RevLineCr, urban_rural


In [ ]:
import pandas as pd
from pathlib import Path
import importlib

# Load df only if it is not already in memory
if "df" not in globals():
    project_root = Path.cwd().resolve().parent
    df = pd.read_csv(project_root / "data" / "train.csv")

df.head()

,id,LoanNr_ChkDgt,Name,City,State,Bank,BankState,ApprovalDate,ApprovalFY,NoEmp,...,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept
0,64afe857c28,9448323000,MIDWEST CRANKSHAFT & ENGINE,HARVEY,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,9-Aug-96,1996,28,...,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0
1,1705a7346c2,2854405007,"Iredesign, Limited",CHICAGO,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,10-Dec-07,2008,1,...,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1
2,7439801ad8a,9300423010,PHILLY'S INC.,ROCHELLE,IL,BMO HARRIS BK NATL ASSOC,IL,23-May-96,1996,6,...,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1
3,a3f8f9d0611,4349265000,USA Laser Imaging Inc.,Loves park,IL,ALPINE BANK & TRUST CO.,IL,4-Nov-10,2011,5,...,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1
4,71e4f243b5d,2433905006,"Dan Morrell, Inc.",LISLE,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,3-May-07,2007,3,...,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0


### Base Cleanup
remove: 'id', 'LoanNr_ChkDgt', 'Name'
### City
cleanup
### Bank
cleanup

### IsLocalBank
created

### State
Removed
### BankState
Removed

In [13]:
base_cleaning = importlib.reload(base_cleaning)
df = base_cleaning.clean_base_columns(df)
df.head()

,City,Bank,ApprovalDate,ApprovalFY,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank
0,HARVEY,JPMORGAN CHASE BANK NATL ASSOC,9-Aug-96,1996,28,1.0,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0,1
1,CHICAGO,JPMORGAN CHASE BANK NATL ASSOC,10-Dec-07,2008,1,2.0,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1,1
2,ROCHELLE,BMO HARRIS BK NATL ASSOC,23-May-96,1996,6,2.0,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1,1
3,LOVES PARK,ALPINE BANK & TRUST CO.,4-Nov-10,2011,5,2.0,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1,1
4,LISLE,JPMORGAN CHASE BANK NATL ASSOC,3-May-07,2007,3,1.0,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0,1


###  NoEmp 
* "raw" -> en caso de árboles.( does nothing) <br>
* "log" -> En caso de KNN o SVM for a  logarithmic transformation of NoEmp using log1p <br>
* "binning" -> (creates bins)Para análisis interpretativo, probar en arboles también para ver si mejora o empeora el rendimiento.

### NoEmp_Log
created with log, removes <b>NoEmp</b>

### NoEmp_Bin
created with binning, removes <b>NoEmp</b>

In [ ]:
# noemp_option = "raw"   # o "log" o "binning"
noemp_option = "log"
# noemp_option = "binning"

noemp = importlib.reload(noemp)
df = noemp.preprocess_noemp(df, option=noemp_option)
df.head()

,City,Bank,ApprovalDate,ApprovalFY,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log,NoEmp_Bin
0,HARVEY,JPMORGAN CHASE BANK NATL ASSOC,9-Aug-96,1996,1.0,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0,0,3.367296,3
1,CHICAGO,JPMORGAN CHASE BANK NATL ASSOC,10-Dec-07,2008,2.0,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1,0,0.693147,1
2,ROCHELLE,BMO HARRIS BK NATL ASSOC,23-May-96,1996,2.0,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1,0,1.945910,2
3,LOVES PARK,ALPINE BANK & TRUST CO.,4-Nov-10,2011,2.0,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1,0,1.791759,1
4,LISLE,JPMORGAN CHASE BANK NATL ASSOC,3-May-07,2007,1.0,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0,0,1.386294,1


In [ ]:
### City y Bank

In [4]:

city_bank = importlib.reload(city_bank)
encoder = city_bank.get_city_bank_encoder()
df_encoded = encoder.fit_transform(df)
df_encoded.head()

NameError: name 'importlib' is not defined

### NewExist

In [5]:
# calling src\preprocessing\newExists.py

# Reload module to pick up latest code changes
newExists = importlib.reload(newExists)

# Choose NewExist preprocessing path: 'A' or 'B'
newexist_option = "A" #===> is_new_business +  newexist_missing_or_invalid
# Option A: Create 'is_new_business' and 'newexist_missing_or_invalid' columns
# Option B: Create only 'is_new_business' column, and delete rows with missing/invalid values

# Apply preprocessing from the imported module
df_newexist = newExists.preprocess_newexist(
    df=df,
    option=newexist_option,
    source_col="NewExist",
)

print(f"NewExist option used: {newexist_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_newexist):,}")

# Quick check of the generated columns
cols_to_show = ["NewExist", "is_new_business"]
if "newexist_missing_or_invalid" in df_newexist.columns:
    cols_to_show.append("newexist_missing_or_invalid")

display(df_newexist[cols_to_show].head(10))

# Show only rows flagged as missing/invalid in Option A
if "newexist_missing_or_invalid" in df_newexist.columns:
    display(
        df_newexist.loc[
            df_newexist["newexist_missing_or_invalid"] == 1,
            ["NewExist", "is_new_business", "newexist_missing_or_invalid"],
        ].head(20)
    )

NewExist option used: A
Rows before: 20,768
Rows after: 20,768


,NewExist,is_new_business,newexist_missing_or_invalid
0,1.0,0,0
1,2.0,1,0
2,2.0,1,0
3,2.0,1,0
4,1.0,0,0
5,2.0,1,0
6,1.0,0,0
7,1.0,0,0
8,1.0,0,0
9,1.0,0,0


,NewExist,is_new_business,newexist_missing_or_invalid
1080,0.0,0,1
2995,0.0,0,1
3226,0.0,0,1
3635,0.0,0,1
4503,0.0,0,1
6016,0.0,0,1
6630,0.0,0,1
10090,0.0,0,1
10473,<NA>,0,1
10730,<NA>,0,1


In [6]:
# calling src\preprocessing\newExists.py

# Reload module to pick up latest code changes
newExists = importlib.reload(newExists)

# Choose NewExist preprocessing path: 'A' or 'B'
newexist_option = "B"

# Apply preprocessing from the imported module
df_newexist = newExists.preprocess_newexist(
    df=df,
    option=newexist_option,
    source_col="NewExist",
)

print(f"NewExist option used: {newexist_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_newexist):,}")

# Quick check of the generated columns
cols_to_show = ["NewExist", "is_new_business"]
if "newexist_missing_or_invalid" in df_newexist.columns:
    cols_to_show.append("newexist_missing_or_invalid")

display(df_newexist[cols_to_show].head(10))

# Show only rows flagged as missing/invalid in Option A
if "newexist_missing_or_invalid" in df_newexist.columns:
    display(
        df_newexist.loc[
            df_newexist["newexist_missing_or_invalid"] == 1,
            ["NewExist", "is_new_business", "newexist_missing_or_invalid"],
        ].head(20)
    )

NewExist option used: B
Rows before: 20,768
Rows after: 20,765


,NewExist,is_new_business
0,1.0,0
1,2.0,1
2,2.0,1
3,2.0,1
4,1.0,0
5,2.0,1
6,1.0,0
7,1.0,0
8,1.0,0
9,1.0,0


### CreateJob

In [7]:
# calling src\preprocessing\createJob.py

# Reload module to pick up latest code changes
createJob = importlib.reload(createJob)

# Choose CreateJob preprocessing path: 'A' (normalize) or 'B' (standardize)
createjob_option = "A"

# Apply preprocessing from the imported module
df_createjob = createJob.preprocess_createjob(
    df=df,
    option=createjob_option,
    source_col="CreateJob",
)

print(f"CreateJob option used: {createjob_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_createjob):,}")

# Quick check of generated columns
cols_to_show = ["CreateJob"]
if "createjob_normalized" in df_createjob.columns:
    cols_to_show.append("createjob_normalized")
if "createjob_standardized" in df_createjob.columns:
    cols_to_show.append("createjob_standardized")

display(df_createjob[cols_to_show].head(10))

CreateJob option used: A
Rows before: 20,768
Rows after: 20,768


,CreateJob,createjob_normalized
0,0,0.0
1,1,0.000114
2,0,0.0
3,0,0.0
4,1,0.000114
5,0,0.0
6,1,0.000114
7,0,0.0
8,6,0.000682
9,1,0.000114


In [8]:
# calling src\preprocessing\createJob.py

# Reload module to pick up latest code changes
createJob = importlib.reload(createJob)

# Choose CreateJob preprocessing path: 'A' (normalize) or 'B' (standardize)
createjob_option = "B"

# Apply preprocessing from the imported module
df_createjob = createJob.preprocess_createjob(
    df=df,
    option=createjob_option,
    source_col="CreateJob",
)

print(f"CreateJob option used: {createjob_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_createjob):,}")

# Quick check of generated columns
cols_to_show = ["CreateJob"]
if "createjob_normalized" in df_createjob.columns:
    cols_to_show.append("createjob_normalized")
if "createjob_standardized" in df_createjob.columns:
    cols_to_show.append("createjob_standardized")

display(df_createjob[cols_to_show].head(10))

CreateJob option used: B
Rows before: 20,768
Rows after: 20,768


,CreateJob,createjob_standardized
0,0,-0.03393
1,1,-0.028997
2,0,-0.03393
3,0,-0.03393
4,1,-0.028997
5,0,-0.03393
6,1,-0.028997
7,0,-0.03393
8,6,-0.004332
9,1,-0.028997


### RetainedJob

In [11]:
# calling src\preprocessing\retainedJob.py

# Reload module to pick up latest code changes
retainedJob = importlib.reload(retainedJob)

# Choose RetainedJob preprocessing path: 'A' (normalize) or 'B' (standardize)
retainedjob_option = "A"
# retainedjob_option = "B"

# Apply preprocessing from the imported module
df_retainedjob = retainedJob.preprocess_retainedjob(
    df=df,
    option=retainedjob_option,
    source_col="RetainedJob",
)

print(f"RetainedJob option used: {retainedjob_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_retainedjob):,}")

# Quick check of generated columns
cols_to_show = ["RetainedJob"]
if "retainedjob_normalized" in df_retainedjob.columns:
    cols_to_show.append("retainedjob_normalized")
if "retainedjob_standardized" in df_retainedjob.columns:
    cols_to_show.append("retainedjob_standardized")

display(df_retainedjob[cols_to_show].head(10))

RetainedJob option used: A
Rows before: 20,768
Rows after: 20,768


,RetainedJob,retainedjob_normalized
0,0,0.0
1,1,0.000114
2,0,0.0
3,5,0.000568
4,3,0.000341
5,0,0.0
6,4,0.000455
7,0,0.0
8,0,0.0
9,6,0.000682


In [10]:
# calling src\preprocessing\createJob.py

# Reload module to pick up latest code changes
createJob = importlib.reload(createJob)

# Choose CreateJob preprocessing path: 'A' (normalize) or 'B' (standardize)
createjob_option = "B"

# Apply preprocessing from the imported module
df_createjob = createJob.preprocess_createjob(
    df=df,
    option=createjob_option,
    source_col="CreateJob",
)

print(f"CreateJob option used: {createjob_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_createjob):,}")

# Quick check of generated columns
cols_to_show = ["CreateJob"]
if "createjob_normalized" in df_createjob.columns:
    cols_to_show.append("createjob_normalized")
if "createjob_standardized" in df_createjob.columns:
    cols_to_show.append("createjob_standardized")

display(df_createjob[cols_to_show].head(10))

CreateJob option used: B
Rows before: 20,768
Rows after: 20,768


,CreateJob,createjob_standardized
0,0,-0.03393
1,1,-0.028997
2,0,-0.03393
3,0,-0.03393
4,1,-0.028997
5,0,-0.03393
6,1,-0.028997
7,0,-0.03393
8,6,-0.004332
9,1,-0.028997


### ApprovalDate and ApprovalFY

In [12]:
import importlib
import sys
from pathlib import Path

# 1. Asegurar que el notebook encuentre la carpeta src (estilo de tu equipo)
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# 2. Importar tus módulos
from preprocessing import approvalFY
from preprocessing import approvalDate


# =======================================================
# ApprovalFY
# =======================================================
approvalFY = importlib.reload(approvalFY)
approvalfy_option = "A"

df_approvalfy = approvalFY.preprocess_approvalfy(
    df=df,
    option=approvalfy_option,
    source_col="ApprovalFY",
)

print(f"Opción de ApprovalFY utilizada: {approvalfy_option}")
print(f"Filas antes: {len(df):,}")
print(f"Filas después: {len(df_approvalfy):,}")

if approvalfy_option == "A":
    print("\nOpción A seleccionada: La columna 'ApprovalFY' fue eliminada.")
    display(df_approvalfy.head(5))
elif approvalfy_option == "B":
    display(df_approvalfy[["ApprovalFY"]].head(10))


# =======================================================
# ApprovalDate
# =======================================================
approvalDate = importlib.reload(approvalDate)
approvaldate_option = "A"

df_approvaldate = approvalDate.preprocess_approvaldate(
    df=df, # ATENCIÓN: Usamos el df base para probar independientemente
    option=approvaldate_option,
    source_col="ApprovalDate",
)

print(f"\nOpción de ApprovalDate utilizada: {approvaldate_option}")
print(f"Filas antes: {len(df):,}")
print(f"Filas después: {len(df_approvaldate):,}")

if approvaldate_option == "A":
    cols_to_show = ["ApprovalYear", "ApprovalMonth"]
    print("\nOpción A seleccionada: Se crearon 'ApprovalYear' y 'ApprovalMonth'.")
    display(df_approvaldate[cols_to_show].head(10))
elif approvaldate_option == "B":
    cols_to_show = ["ApprovalDate"] 
    display(df_approvaldate[cols_to_show].head(10))

Opción de ApprovalFY utilizada: A
Filas antes: 20,768
Filas después: 20,768

Opción A seleccionada: La columna 'ApprovalFY' fue eliminada.


,id,LoanNr_ChkDgt,Name,City,State,Bank,BankState,ApprovalDate,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept
0,64afe857c28,9448323000,MIDWEST CRANKSHAFT & ENGINE,HARVEY,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,9-Aug-96,28,1.0,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0
1,1705a7346c2,2854405007,"Iredesign, Limited",CHICAGO,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,10-Dec-07,1,2.0,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1
2,7439801ad8a,9300423010,PHILLY'S INC.,ROCHELLE,IL,BMO HARRIS BK NATL ASSOC,IL,23-May-96,6,2.0,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1
3,a3f8f9d0611,4349265000,USA Laser Imaging Inc.,Loves park,IL,ALPINE BANK & TRUST CO.,IL,4-Nov-10,5,2.0,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1
4,71e4f243b5d,2433905006,"Dan Morrell, Inc.",LISLE,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,3-May-07,3,1.0,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0



Opción de ApprovalDate utilizada: A
Filas antes: 20,768
Filas después: 20,768

Opción A seleccionada: Se crearon 'ApprovalYear' y 'ApprovalMonth'.


,ApprovalYear,ApprovalMonth
0,1996,8
1,2007,12
2,1996,5
3,2010,11
4,2007,5
5,2003,3
6,2005,10
7,1993,2
8,1999,5
9,2004,3


In [ ]:
FranchiseCode

In [16]:
# llamando a src\preprocessing\franchise_code.py

# Primero importar el módulo
import src.preprocessing.franchise_code as franchise_code

# Recargar el módulo para aplicar los últimos cambios en el código
import importlib
franchise_code = importlib.reload(franchise_code)

# Elegir la ruta de preprocesamiento de FranchiseCode: 'binary' o 'raw'
franchise_option = "binary" 

# Aplicar el preprocesamiento desde el módulo importado
df_franchise = franchise_code.preprocess_franchise_code(
    df=df,
    option=franchise_option,
    source_col="FranchiseCode",
)

print(f"Opción de FranchiseCode utilizada: {franchise_option}")
print(f"Filas antes: {len(df):,}")
print(f"Filas después: {len(df_franchise):,}")

# Comprobación rápida de las columnas generadas
cols_to_show = []
if "FranchiseCode" in df_franchise.columns:
    cols_to_show.append("FranchiseCode")
if "IsFranchise" in df_franchise.columns:
    cols_to_show.append("IsFranchise")

display(df_franchise[cols_to_show].head(10))


Opción de FranchiseCode utilizada: binary
Filas antes: 20,768
Filas después: 20,768


,IsFranchise
0,0
1,0
2,0
3,0
4,0
5,1
6,0
7,1
8,0
9,0


In [ ]:
UrbanRural

In [17]:
# llamando a src\preprocessing\urban_rural.py

# Primero importar el módulo
import src.preprocessing.urban_rural as urban_rural

# Recargar el módulo para aplicar los últimos cambios en el código
import importlib
urban_rural = importlib.reload(urban_rural)

# Elegir la ruta de preprocesamiento de UrbanRural: 'onehot' o 'text'
urbanrural_option = "onehot" 

# Aplicar el preprocesamiento desde el módulo importado
df_urbanrural = urban_rural.preprocess_urban_rural(
    df=df,
    option=urbanrural_option,
    source_col="UrbanRural",
)

print(f"Opción de UrbanRural utilizada: {urbanrural_option}")
print(f"Filas antes: {len(df):,}")
print(f"Filas después: {len(df_urbanrural):,}")

# Comprobación rápida de las columnas generadas
cols_to_show = []
if "UrbanRural" in df_urbanrural.columns:
    cols_to_show.append("UrbanRural")

# Añadir las columnas one-hot generadas si existen
zone_cols = [col for col in df_urbanrural.columns if col.startswith('Zone_')]
cols_to_show.extend(zone_cols)

display(df_urbanrural[cols_to_show].head(10))

Opción de UrbanRural utilizada: onehot
Filas antes: 20,768
Filas después: 20,768


,Zone_Rural,Zone_Undefined,Zone_Urban
0,0,1,0
1,0,0,1
2,0,1,0
3,0,0,1
4,0,0,1
5,0,0,1
6,0,0,1
7,0,1,0
8,0,1,0
9,0,0,1


# RevLineCr

In [ ]:
# calling src\preprocessing\RevLineCr.py

import src.preprocessing.RevLineCr as RevLineCr
import importlib

# Reload module to pick up latest code changes
RevLineCr = importlib.reload(RevLineCr)

# Choose RevLineCr preprocessing path: 'A' or 'B'
revlinecr_option = "A"

# Apply preprocessing from the imported module
df_revlinecr = RevLineCr.preprocess_revlinecr(
    df=df,
    option=revlinecr_option,
    source_col="RevLineCr",
)

print(f"RevLineCr option used: {revlinecr_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_revlinecr):,}")

# Quick check of generated columns
cols_to_show = ["RevLineCr"]

if "revlinecr_is_nonstandard" in df_revlinecr.columns:
    cols_to_show.append("revlinecr_is_nonstandard")
if "revlinecr_is_missing" in df_revlinecr.columns:
    cols_to_show.append("revlinecr_is_missing")

rev_cols = [col for col in df_revlinecr.columns if col.startswith("revlinecr_")]
cols_to_show.extend(rev_cols)

display(df_revlinecr[cols_to_show].head(10))

In [ ]:
# calling src\preprocessing\RevLineCr.py

import src.preprocessing.RevLineCr as RevLineCr
import importlib

RevLineCr = importlib.reload(RevLineCr)

revlinecr_option = "B"

df_revlinecr = RevLineCr.preprocess_revlinecr(
    df=df,
    option=revlinecr_option,
    source_col="RevLineCr",
)

print(f"RevLineCr option used: {revlinecr_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_revlinecr):,}")

cols_to_show = ["RevLineCr"]

rev_cols = [col for col in df_revlinecr.columns if col.startswith("revlinecr_")]
cols_to_show.extend(rev_cols)

display(df_revlinecr[cols_to_show].head(10))

# LowDoc

In [ ]:
# calling src\preprocessing\LowDoc.py

import src.preprocessing.LowDoc as LowDoc
import importlib

LowDoc = importlib.reload(LowDoc)

lowdoc_option = "A"

df_lowdoc = LowDoc.preprocess_lowdoc(
    df=df,
    option=lowdoc_option,
    source_col="LowDoc",
)

print(f"LowDoc option used: {lowdoc_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_lowdoc):,}")

cols_to_show = ["LowDoc"]

if "lowdoc_is_nonstandard" in df_lowdoc.columns:
    cols_to_show.append("lowdoc_is_nonstandard")
if "lowdoc_is_missing" in df_lowdoc.columns:
    cols_to_show.append("lowdoc_is_missing")

lowdoc_cols = [col for col in df_lowdoc.columns if col.startswith("lowdoc_")]
cols_to_show.extend(lowdoc_cols)

display(df_lowdoc[cols_to_show].head(10))

In [ ]:
# calling src\preprocessing\LowDoc.py

import src.preprocessing.LowDoc as LowDoc
import importlib

LowDoc = importlib.reload(LowDoc)

lowdoc_option = "B"

df_lowdoc = LowDoc.preprocess_lowdoc(
    df=df,
    option=lowdoc_option,
    source_col="LowDoc",
)

print(f"LowDoc option used: {lowdoc_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_lowdoc):,}")

cols_to_show = ["LowDoc"]

lowdoc_cols = [col for col in df_lowdoc.columns if col.startswith("lowdoc_")]
cols_to_show.extend(lowdoc_cols)

display(df_lowdoc[cols_to_show].head(10))